This is a starter notebook for an updated module 5 of ML Zoomcamp

The code is based on the modules 3 and 4. We use the same dataset: [telco customer churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

In [1]:
import pandas as pd
import numpy as np
import sklearn
import pickle

from sklearn.pipeline import make_pipeline

In [2]:
print(f'pandas=={pd.__version__}')
print(f'numpy=={np.__version__}')
print(f'sklearn=={sklearn.__version__}')

pandas==3.0.2
numpy==2.4.4
sklearn==1.8.0


In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [4]:
data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data_url)

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn.str.lower() == 'yes').astype(int)

In [5]:
y_train = df.churn

In [6]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [7]:
pipeline = make_pipeline(
    DictVectorizer(),
    LogisticRegression(solver='liblinear')
)

In [10]:
train_dict = df[categorical + numerical].to_dict(orient='records')

X_train = pipeline.fit(train_dict, y_train)

In [11]:
## saving a model with pickle
with open('model.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

In [12]:
## loading a model with pickle
with open('model.bin', 'rb') as f_in:
    pipeline = pickle.load(f_in)

In [13]:
customer = {
    'gender': 'Male',
    'seniorcitizen': 0,
    'partner': 'No',
    'dependents': 'Yes',
    'phoneservice': 'No',
    'multiplelines': 'No phone service',
    'internetservice': 'DSL',
    'onlinesecurity': 'No',
    'onlinebackup': 'Yes',
    'deviceprotection': 'No',
    'techsupport': 'No',
    'streamingtv': 'No',
    'streamingmovies': 'No',
    'contract': 'Month-to-month',
    'paperlessbilling': 'Yes',
    'paymentmethod': 'Electronic check',
    'tenure': 6,
    'monthlycharges': 29.85,
    'totalcharges': 129.85
}

churn = pipeline.predict_proba(customer)[0,1]


print(f'probability of churning = {churn}')
if churn >= 0.5:
    print('send email with promo')
else:
    print("don't do anything")

probability of churning = 0.541558136103048
send email with promo
